In [23]:
import numpy as np
import pandas as pd
import datetime

In [24]:
capston_df = pd.read_csv('dataset/rawdata/Sales_Dataset.csv')
capston_df.head()

,Order ID,Amount,Profit,Quantity,Category,Sub-Category,PaymentMode,Order Date,CustomerName,State,City,Year-Month
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,Miami,2023-06
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2024-12
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,Buffalo,2021-07
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,Miami,2023-06
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2024-12


In [25]:
capston_df.columns

Index(['Order ID', 'Amount', 'Profit', 'Quantity', 'Category', 'Sub-Category',
       'PaymentMode', 'Order Date', 'CustomerName', 'State', 'City',
       'Year-Month'],
      dtype='object')

In [26]:
capston_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Order ID      1194 non-null   object
 1   Amount        1194 non-null   int64 
 2   Profit        1194 non-null   int64 
 3   Quantity      1194 non-null   int64 
 4   Category      1194 non-null   object
 5   Sub-Category  1194 non-null   object
 6   PaymentMode   1194 non-null   object
 7   Order Date    1194 non-null   object
 8   CustomerName  1194 non-null   object
 9   State         1194 non-null   object
 10  City          1194 non-null   object
 11  Year-Month    1194 non-null   object
dtypes: int64(3), object(9)
memory usage: 112.1+ KB


In [27]:
# Standardise the columns
capston_df.columns = (
    capston_df.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=True)
    .str.replace('-', '_', regex=True)
)
capston_df.head()

,order_id,amount,profit,quantity,category,sub_category,paymentmode,order_date,customername,state,city,year_month
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,Miami,2023-06
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2024-12
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,Buffalo,2021-07
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,Miami,2023-06
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2024-12


In [28]:
# Check for duplicate row
# capston_df.duplicated().any() 
# capston_df.duplicated().sum() # Count how many duplicate
# capston_df[capston_df.duplicated()] # Display DUplicated rows

### Normalising the data (Building Dimensional Tables)

In [29]:
# city_df table
city_df = capston_df[['city', 'state']].reset_index(drop=True)
city_df['city_id'] = city_df.index + 1
city_df = city_df[['city_id', 'city', 'state']]

city_df

,city_id,city,state
0,1,Miami,Florida
1,2,Chicago,Illinois
2,3,Buffalo,New York
3,4,Miami,Florida
4,5,Chicago,Illinois
...,...,...,...
1189,1190,New York City,New York
1190,1191,Rochester,New York
1191,1192,Austin,Texas
1192,1193,Buffalo,New York


In [45]:
# Customer_df Table (with city as foreign key)

# customer_df = (
#     capston_df[['customername', 'city', 'state']]
#     .drop_duplicates()
#     .reset_index(drop=True)
# )
# # merge with city_df to get primary key
# customer_df = customer_df.merge(city_df, on=['city', 'state'], how='left')
# # Assign a new primary key (customer_id)
# customer_df['customer_id'] = customer_df.index + 1
# # Keep all required column
# customer_df = customer_df[['customer_id', 'customername', 'city_id']]
customer_df = capston_df[['customername']].reset_index(drop=True)
customer_df['customer_id'] = customer_df.index + 1
customer_df['city_id'] = np.random.choice(city_df['city_id'], size=len(customer_df))
customer_df = customer_df[['customer_id', 'customername', 'city_id']]

customer_df

,customer_id,customername,city_id
0,1,David Padilla,903
1,2,Connor Morgan,514
2,3,Robert Stone,714
3,4,David Padilla,382
4,5,Connor Morgan,707
...,...,...,...
1189,1190,Megan Mclean,847
1190,1191,Caitlin Hunt,719
1191,1192,Jenna Holland,716
1192,1193,Stephanie Oconnell,796


In [31]:
# payment_df table
payment_df = capston_df[['paymentmode']].drop_duplicates().reset_index(drop=True)
payment_df['payment_id'] = payment_df.index + 1
payment_df = payment_df[['payment_id', 'paymentmode']]

payment_df

,payment_id,paymentmode
0,1,UPI
1,2,Debit Card
2,3,EMI
3,4,Credit Card
4,5,COD


In [32]:
# product_df table

product_df = capston_df[['category', 'sub_category']].drop_duplicates().reset_index(drop=True)
product_df['product_id'] = product_df.index + 1
product_df = product_df[['product_id', 'category', 'sub_category']]

product_df


,product_id,category,sub_category
0,1,Electronics,Electronic Games
1,2,Electronics,Printers
2,3,Office Supplies,Pens
3,4,Electronics,Laptops
4,5,Furniture,Tables
5,6,Furniture,Chairs
6,7,Office Supplies,Markers
7,8,Furniture,Sofas
8,9,Office Supplies,Paper
9,10,Office Supplies,Binders


In [33]:
# Convert to Date Format
capston_df['order_date'] = pd.to_datetime(capston_df['order_date'], format='mixed') # formatting datatype of order_date column to datetime 

capston_df['year'] = capston_df['order_date'].dt.year
capston_df['month'] = capston_df['order_date'].dt.month
capston_df['day'] = capston_df['order_date'].dt.day

# date_df table
date_df = capston_df[['order_date', 'year_month', 'year', 'month', 'day']].reset_index(drop=True)
date_df['date_id'] = date_df.index + 1
date_df = date_df[['date_id', 'order_date', 'year', 'month', 'day']]

date_df


,date_id,order_date,year,month,day
0,1,2023-06-27,2023,6,27
1,2,2024-12-27,2024,12,27
2,3,2021-07-25,2021,7,25
3,4,2023-06-27,2023,6,27
4,5,2024-12-27,2024,12,27
...,...,...,...,...,...
1189,1190,2024-07-31,2024,7,31
1190,1191,2020-06-02,2020,6,2
1191,1192,2022-12-15,2022,12,15
1192,1193,2020-08-07,2020,8,7


In [ ]:
order_fact_df = capston_df[['amount', 'profit', 'quantity']].reset_index(drop=True)
order_fact_df['order_id'] = order_fact_df.index + 1

order_fact_df['product_id'] = np.random.choice(product_df['product_id'], size=len(order_fact_df))
order_fact_df['customer_id'] = np.random.choice(customer_df['customer_id'], size=len(order_fact_df))
order_fact_df['date_id'] = np.random.choice(date_df['date_id'], size=len(order_fact_df))
order_fact_df['payment_id'] = np.random.choice(payment_df['payment_id'], size=len(order_fact_df))

order_fact_df['amount'] = order_fact_df['amount'].round(2)
order_fact_df['profit'] = order_fact_df['profit'].round(2)
order_fact_df = order_fact_df[['order_id', 'amount', 'profit', 'quantity', 'product_id', 'customer_id', 'date_id', 'payment_id']]

# order_fact_df





# # Merge with dimesions using consistent keys
# order_fact_df = order_fact_df.merge(product_df[['category', 'product_id']], on='category', how='left')
# order_fact_df = order_fact_df.merge(customer_df[['customername', 'customer_id']], on='customername', how='left')
# order_fact_df = order_fact_df.merge(date_df[['order_date', 'date_id']], on='order_date', how='left')
# order_fact_df = order_fact_df.merge(payment_df[['paymentmode', 'payment_id']], on='paymentmode', how='left')

# # Create the fact table columns
# order_fact_df['order_id'] = order_fact_df.index + 1
# order_fact_df = order_fact_df[['order_id', 'product_id', 'customer_id', 'date_id', 'payment_id',
#                                'amount', 'profit', 'quantity']]

# # check for duplicates
# print("Row count: ", len(order_fact_df))
# print("unique order IDs", order_fact_df['order_id'].nunique())


# order_fact_df
# order_fact_df = capston_df[['amount', 'profit', 'quantity']].drop_duplicates().reset_index(drop=True)

# order_fact_df = capston_df.merge(product_df, on='category', how='left')
# order_fact_df = order_fact_df.merge(customer_df, on='customername',  how='left')
# order_fact_df = order_fact_df.merge(date_df, on='order_date', how='left')
# order_fact_df = order_fact_df.merge(payment_df, on='paymentmode', how='left')

# order_fact_df['order_id'] = order_fact_df.index + 1 
# order_fact_df = order_fact_df[['order_id', 'product_id', 'customer_id', 'date_id', 'payment_id', 'customername', 'profit', 'quantity']]

# order_fact_df

# order_fact_df = capston_df[['amount', 'profit', 'quantity']].reset_index(drop=True)
# order_fact_df['order_id'] = order_fact_df.index + 1
# order_fact_df['amount'] = order_fact_df['amount'].round(2)
# order_fact_df['profit'] = order_fact_df['profit'].round(2)
# order_fact_df[['amount', 'profit', 'quantity', 'product_id']]

Row count:  1194
unique order IDs 1194


In [50]:
# Check if duplicates exist
order_fact_df.duplicated(subset=['product_id','customer_id','date_id','payment_id']).sum()

0

In [51]:
# Compare with original data
print("Original capston_df:", len(capston_df))
print("Order Fact Table:", len(order_fact_df))

Original capston_df: 1194
Order Fact Table: 1194


### Data insertion (to_sql() via SQLAlchemy)

In [53]:
# Database connection
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

uid = 'postgres'
pwd = 'password'
server = 'localhost'
database = 'capstone'

engine = create_engine(f'postgresql://{uid}:{pwd}@{server}:5432/{database}')

In [54]:
# Writing Pandas Table into SQL Database
order_fact_df.to_sql("orders_fact", engine, if_exists='replace', index=False)
product_df.to_sql("product_category", engine, if_exists='replace', index=False)
customer_df.to_sql("customers", engine, if_exists='replace', index=False)
city_df.to_sql("cities", engine, if_exists='replace', index=False)
payment_df.to_sql("payment_df", engine, if_exists='replace', index=False)
date_df.to_sql("date", engine, if_exists='replace', index='False')


194

### Save data to csv file

In [56]:
date_df.to_csv(r'dataset/cleaneddata/date.csv')
payment_df.to_csv(r'dataset/cleaneddata/payment.csv')
city_df.to_csv(r'dataset/cleaneddata/city.csv')
customer_df.to_csv(r'dataset/cleaneddata/customers.csv')
order_fact_df.to_csv(r'dataset/cleaneddata/orders_fact.csv')